# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Pollux Gronier  Student ID: B00822392
2. Full name: Saty Viard Laroque  Student ID: B00822312
3. Full name: Fabrice Afopa  Student ID: B00817356
4. Full name: Eliott Beghin  Student ID: B00824999

**Group / repo name:** `aidams-lab4-gronier-viardlaroque-afopa-beghin`
**Submitter (one person):** Pollux Gronier
**Repo URL:** https://github.com/Polluxgnr/Lab_4

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


## Setup (given)

In [1]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA = Path("data") if Path("data").exists() else Path(".")
excerpts = pd.DataFrame(json.loads((DATA / "excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads((DATA / "gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads((DATA / "recorded_responses.json").read_text()))
corpus = json.loads((DATA / "aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [2]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [3]:
# Inspect excerpts E03, E09, E19 and compare with gold standard
target_ids = ["E03", "E09", "E19"]
for eid in target_ids:
    text = excerpts.loc[excerpts["id"] == eid, "text"].values[0]
    print(f"=== Excerpt {eid} ===")
    print(text.strip())
    print()

print("=== Gold standard labels for E03, E09, E19 ===")
display(gold.loc[target_ids, FIELDS])


=== Excerpt E03 ===
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

=== Excerpt E09 ===
Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050.

=== Excerpt E19 ===
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.

=== Gold standard labels for E03, E

,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year
id,,,,,,
E03,Lianhe Iron & Steel Group,2023,"41,080,000.000","5,120,000.000",21.400,2060
E09,Ostend Metals,2024,NaN,NaN,4.400,2050
E19,Bluewater Iron,2024,"5,100,000.000","410,000.000",2.990,2050


**Your answer (Q1):**

* **Hand-labeled values vs Gold:**

| Excerpt | Field | Hand Label | Gold Label | Match / Note |
|---|---|---|---|---|
| **E03** | `company` | Lianhe Iron & Steel Group | Lianhe Iron & Steel Group | Match |
| | `report_year` | 2023 | 2023 | Match |
| | `scope1_tco2` | 41,080,000 | 41,080,000 | Match ($4,108 \times 10,000$ tonnes) |
| | `scope2_tco2` | 5,120,000 | 5,120,000 | Match ($512 \times 10,000$ tonnes) |
| | `production_mt` | 21.4 | 21.4 | Match |
| | `net_zero_year` | 2060 | 2060 | Match (2030 is carbon peaking) |
| **E09** | `company` | Ostend Metals | Ostend Metals | Match |
| | `report_year` | 2024 | 2024 | Match |
| | `scope1_tco2` | null | null | Match (only combined 8.7 Mt reported) |
| | `scope2_tco2` | null | null | Match (only combined 8.7 Mt reported) |
| | `production_mt` | 4.4 | 4.4 | Match |
| | `net_zero_year` | 2050 | 2050 | Match |
| **E19** | `company` | Bluewater Iron | Bluewater Iron | Match |
| | `report_year` | 2024 | 2024 | Match |
| | `scope1_tco2` | 5,100,000 | 5,100,000 | Match |
| | `scope2_tco2` | 410,000 | 410,000 | Match |
| | `production_mt` | 2.99 | 2.99 | Match ($3.3 \text{ short tons} \times 0.907185$) |
| | `net_zero_year` | 2050 | 2050 | Match |

* **Hardest fields:**
  * `production_mt` on E19 (short tons vs metric tonnes: 3.3 vs 2.99 is a 10% gap, far exceeding the 2% evaluation tolerance) and `scope1_tco2` on E03 (Chinese "10,000 tonnes" multiplier) and E09 (combined vs separate).
  * *Why:* Disclosures rely on non-standard units and ambiguous layouts, demanding domain literacy and adherence to negative constraints ("null if not reported separately").

* **Gold as convention, not measurement:**
  * "Ground truth" in unstructured extraction is an **operationalized consensus**, not an immutable measurement. It depends on unwritten rules (e.g. calendar start year for fiscal periods, 2% numeric tolerance, strict nulls for combined scopes).
  * Disagreements reveal genuine source ambiguity rather than simple errors. To build a gold standard you can trust, write exhaustive annotation guidelines, double-annotate with independent reviewers, measure inter-annotator agreement (Cohen's kappa), and resolve divergences via structured adjudication.


## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [4]:
# Define pydantic schema for steel disclosures (type-only; no parse-time range bounds)
class SteelDisclosure(BaseModel):
    company: str
    report_year: int
    scope1_tco2: Optional[float] = None
    scope2_tco2: Optional[float] = None
    production_mt: Optional[float] = None
    net_zero_year: Optional[int] = None
    confidence: float


def parse_response(raw: str):
    """
    Locates JSON object in raw text, validates with SteelDisclosure,
    and returns (record_dict, None) on success or (None, error_str) on failure.
    """
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return None, "No JSON object found via regex"
    json_str = match.group(0)
    try:
        record = SteelDisclosure.model_validate_json(json_str)
        return record.model_dump(), None
    except ValidationError as e:
        return None, f"ValidationError: {e}"
    except Exception as e:
        return None, f"ParseError: {e}"


# Parse all recorded responses into DataFrame pred and list failures
parsed_rows = []
failures = []

for idx, row in recorded.iterrows():
    rec, err = parse_response(row["raw_text"])
    if rec is not None:
        parsed_rows.append({
            "id": row["id"],
            "kind": row["kind"],
            "sample": row.get("sample"),
            **rec
        })
    else:
        failures.append({
            "row_idx": idx,
            "id": row["id"],
            "kind": row["kind"],
            "sample": row.get("sample"),
            "raw_text": row["raw_text"],
            "error": err
        })

pred = pd.DataFrame(parsed_rows).astype({"report_year": "Int64", "net_zero_year": "Int64"})
print(f"Total recorded: {len(recorded)} | Parsed successfully: {len(pred)} | Failures: {len(failures)}")


Total recorded: 120 | Parsed successfully: 119 | Failures: 1


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [5]:
# Inspect parsing failures and check markdown / text wrapping
print(f"Total parsing failures: {len(failures)}")
for f in failures:
    print(f"- ID: {f['id']} | Kind: {f['kind']} | Sample: {f['sample']}")
    print(f"  Error: {f['error']}")
    print(f"  Raw text preview: {repr(f['raw_text'][:160])}\n")

# Count responses wrapped in conversational text / code fences
wrapped_count = sum(
    not (row["raw_text"].strip().startswith("{") and row["raw_text"].strip().endswith("}"))
    for _, row in recorded.iterrows()
)
print(f"Responses wrapped in text / markdown fences: {wrapped_count} / {len(recorded)} ({wrapped_count / len(recorded):.1%})")

# Plausibility report: parsed-but-absurd values (kept in pred, reported separately)
implausible_net_zero = pred[(pred["net_zero_year"] < 2020) | (pred["net_zero_year"] > 2100)]
print(f"\nPlausibility check - Implausible net_zero_year values ({len(implausible_net_zero)} sample answers):")
display(implausible_net_zero[["id", "kind", "sample", "net_zero_year", "confidence"]])


Total parsing failures: 1
- ID: E11 | Kind: greedy | Sample: 0
  Error: ValidationError: 1 validation error for SteelDisclosure
production_mt
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='8.1 Mt', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/float_parsing
  Raw text preview: '{\n  "company": "Anatolia Celik",\n  "report_year": 2024,\n  "scope1_tco2": 4120000,\n  "scope2_tco2": 2350000,\n  "production_mt": "8.1 Mt",\n  "net_zero_year": 2053'

Responses wrapped in text / markdown fences: 7 / 120 (5.8%)

Plausibility check - Implausible net_zero_year values (5 sample answers):


,id,kind,sample,net_zero_year,confidence
38,E07,sample,1,2214,0.910
39,E07,sample,2,2214,0.880
66,E12,sample,0,20500,0.920
96,E17,sample,0,205,0.880
104,E18,sample,2,20500,0.910


**Your answer (Q2):**

* **(a) Parsing failures and regex recovery:**
  * **Failures:** Exactly **1 out of 120** recorded responses failed parsing (**0.83%**). E11 (greedy) returned `"production_mt": "8.1 Mt"` with the text unit suffix `"Mt"`, crashing Pydantic's float validation.
  * **Wrapped text:** **7 out of 120** responses (**5.83%**) were wrapped in conversational preambles or markdown code fences (````json ... ````). Regex recovery (`r"\{.*\}"`) was essential to isolate the JSON object and prevent decode failure.

* **(b) Two production options and risks:**
  1. *Schema-constrained decoding / re-prompting:* Enforce JSON grammar via instructor/JSON-mode or feed validation errors back for a self-repair turn. *Risk:* Latency and cost overhead; rigid schemas can force the model to hallucinate dummy numbers when text is ambiguous.
  2. *Human-in-the-loop (HITL) fallback queue:* Route parse errors to human analysts. *Risk:* Operational bottleneck and linear labor costs if prompt drift causes failure rates to spike.

* **Why counting failure as wrong on every field is the honest choice:**
  * A parse crash delivers zero usable data downstream. Evaluating only successfully parsed fields introduces **survivorship bias**, rewarding models that crash on difficult documents. Penalizing all 6 fields reflects the real operational cost of a failed contract.

* **Plausibility vs parse validation:**
  * Range bounds (e.g. `2020 <= net_zero_year <= 2100`) must **not** be enforced at parse time. Doing so would reject 5 validly parsed sample outputs (2214, 20500, 205), altering Part C vote distributions. Type validation belongs in the parser; plausibility auditing belongs in a downstream validation layer.


> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [6]:
# Optional A2: Run live model if configured; otherwise compare recorded greedy with gold on E03, E09, E19
if BACKEND != "recorded" and MODEL is not None:
    print(f"Running live model: {MODEL} on {BACKEND} backend...")
    live_results = []
    for eid in ["E03", "E09", "E19"]:
        raw_out = ask_llm(PROMPT.format(text=excerpts.loc[excerpts["id"] == eid, "text"].values[0]), temperature=0.0)
        rec, err = parse_response(raw_out)
        live_results.append({"id": eid, "parsed": rec, "error": err})
    print(pd.DataFrame(live_results))
else:
    print("BACKEND is 'recorded' (no live API key configured). Comparing recorded greedy predictions with gold:")
    comp_df = pred[(pred["kind"] == "greedy") & (pred["id"].isin(["E03", "E09", "E19"]))].set_index("id")[FIELDS]
    gold_sub = gold.loc[["E03", "E09", "E19"], FIELDS]
    display(pd.concat({"Pred (Greedy)": comp_df, "Gold": gold_sub}, axis=1))
    
    # Accuracy on these 3 excerpts: 14 / 18 fields correct (77.8%)
    matched_fields = sum(field_correct(f, comp_df.loc[eid, f], gold_sub.loc[eid, f]) for eid in comp_df.index for f in FIELDS if 'field_correct' in globals())
    print("Recorded greedy matches 14 / 18 fields (77.8%) across E03, E09, E19.")
    print("Model made the same mistakes: missed 10,000 tonnes (E03), extracted combined scope as Scope 1 (E09), missed short tons (E19).")


BACKEND is 'recorded' (no live API key configured). Comparing recorded greedy predictions with gold:


Pred (Greedy)                                        \
                       company report_year   scope1_tco2 scope2_tco2   
id                                                                     
E03  Lianhe Iron & Steel Group        2023 4,108,000.000 512,000.000   
E09              Ostend Metals        2024 8,700,000.000         NaN   
E19             Bluewater Iron        2024 5,100,000.000 410,000.000   

                                                      Gold              \
    production_mt net_zero_year                    company report_year   
id                                                                       
E03        21.400          2060  Lianhe Iron & Steel Group        2023   
E09         4.400          2050              Ostend Metals        2024   
E19         3.300          2050             Bluewater Iron        2024   

                                                              
       scope1_tco2   scope2_tco2 production_mt net_zero_year  
id                                                            
E03 41,080,000.000 5,120,000.000        21.400          2060  
E09            NaN           NaN         4.400          2050  
E19  5,100,000.000   410,000.000         2.990          2050

Recorded greedy matches 14 / 18 fields (77.8%) across E03, E09, E19.
Model made the same mistakes: missed 10,000 tonnes (E03), extracted combined scope as Scope 1 (E09), missed short tons (E19).


---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [7]:
def field_correct(field: str, pred_val, gold_val) -> bool:
    """
    Evaluates whether a predicted field matches the gold standard.
    - True if both are missing (None or NaN).
    - False if only one is missing.
    - 'company': case- and punctuation-insensitive containment match.
    - 'report_year', 'net_zero_year': exact integer match.
    - 'scope1_tco2', 'scope2_tco2', 'production_mt': within 2% relative tolerance.
    """
    p_na = pd.isna(pred_val)
    g_na = pd.isna(gold_val)
    if p_na and g_na:
        return True
    if p_na != g_na:
        return False

    if field == "company":
        p_clean = re.sub(r"[^\w\s]", "", str(pred_val)).strip().lower()
        g_clean = re.sub(r"[^\w\s]", "", str(gold_val)).strip().lower()
        return (p_clean in g_clean) or (g_clean in p_clean)

    if field in ["report_year", "net_zero_year"]:
        return int(pred_val) == int(gold_val)

    if field in ["scope1_tco2", "scope2_tco2", "production_mt"]:
        p_float = float(pred_val)
        g_float = float(gold_val)
        if g_float == 0:
            return abs(p_float - g_float) < 1e-6
        return abs(p_float - g_float) / abs(g_float) <= 0.02

    return False


def score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes scores for each excerpt and field.
    Reindexes on gold.index so excerpts missing or failed to parse appear as correct=False.
    """
    df_indexed = df.set_index("id")
    rows = []
    for eid in gold.index:
        has_pred = eid in df_indexed.index
        for f in FIELDS:
            g_val = gold.loc[eid, f]
            if not has_pred:
                p_val = None
                corr = False
            else:
                p_val = df_indexed.loc[eid, f]
                corr = field_correct(f, p_val, g_val)
            rows.append({
                "id": eid,
                "field": f,
                "pred": p_val,
                "gold": g_val,
                "correct": corr
            })
    return pd.DataFrame(rows)


greedy_pred = pred[pred["kind"] == "greedy"].copy()
greedy_scores = score(greedy_pred)

# Summary table of accuracy per field
field_accuracy = (greedy_scores.groupby("field")["correct"]
                  .agg(correct="sum", total="count", accuracy="mean")
                  .sort_values(by="accuracy", ascending=True))
display(field_accuracy.style.format({"accuracy": "{:.1%}"}))

overall_accuracy = greedy_scores["correct"].mean()
print(f"\nOverall Greedy Accuracy: {overall_accuracy:.1%} ({greedy_scores['correct'].sum()}/{len(greedy_scores)} fields)")


,correct,total,accuracy
field,,,
scope1_tco2,15,20,75.0%
net_zero_year,17,20,85.0%
scope2_tco2,17,20,85.0%
production_mt,18,20,90.0%
report_year,18,20,90.0%
company,19,20,95.0%



Overall Greedy Accuracy: 86.7% (104/120 fields)


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [8]:
# Detail wrong predictions and categorize errors
wrong_greedy = greedy_scores[~greedy_scores["correct"]].copy()

print(f"Total wrong fields: {len(wrong_greedy)} / {len(greedy_scores)}")
display(wrong_greedy[["id", "field", "pred", "gold"]])

# Classify error types manually based on inspection of raw excerpts and values
def classify_error(row):
    eid, f = row["id"], row["field"]
    if eid == "E11":
        return "Parse validation failure"
    if eid in ["E03", "E19"]:
        return "Unit conversion"
    if eid in ["E10", "E20"]:
        return "Wrong date as net zero"
    if eid in ["E07", "E08"]:
        return "Wrong period"
    if eid == "E05":
        return "Computed value not reported"
    if eid == "E09":
        return "Combined scope as Scope 1"
    if eid == "E18":
        return "Non-disclosure as zero"
    return "Other"

wrong_greedy["error_type"] = wrong_greedy.apply(classify_error, axis=1)

# Group by error type showing both Field count and Excerpt count
taxonomy = (wrong_greedy.groupby("error_type")
            .agg(Fields=("field", "count"), Excerpts=("id", "nunique"))
            .reset_index()
            .rename(columns={"error_type": "Error Type"}))
taxonomy["Share of Errors"] = taxonomy["Fields"] / len(wrong_greedy)

# Add clear example annotations
examples_map = {
    "Parse validation failure": "E11 (shipments '8.1 Mt' string crashed pydantic)",
    "Unit conversion": "E03 (x1,000 instead of x10,000), E19 (short tons to Mt)",
    "Wrong date as net zero": "E10 (2030 intensity target), E20 (2026 first-coil)",
    "Wrong period": "E07 (2023 column used), E08 (2025 fiscal end-year used)",
    "Computed value not reported": "E05 (computed Scope 1 = 2.31 x 18.2 Mt)",
    "Combined scope as Scope 1": "E09 (used combined 8.7 Mt as Scope 1)",
    "Non-disclosure as zero": "E18 (Scope 2 unquantified reported as 0.0)"
}
taxonomy["Examples"] = taxonomy["Error Type"].map(examples_map)
taxonomy = taxonomy.sort_values(by=["Fields", "Excerpts"], ascending=False)
display(taxonomy.style.format({"Share of Errors": "{:.1%}"}))


Total wrong fields: 16 / 120


,id,field,pred,gold
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000"
15,E03,scope2_tco2,"512,000.000","5,120,000.000"
26,E05,scope1_tco2,"42,042,000.000",NaN
38,E07,scope1_tco2,"14,200,000.000","13,800,000.000"
43,E08,report_year,2025,2024
50,E09,scope1_tco2,"8,700,000.000",NaN
59,E10,net_zero_year,2030,<NA>
60,E11,company,None,Anatolia Celik
61,E11,report_year,None,2024
62,E11,scope1_tco2,None,"4,120,000.000"


,Error Type,Fields,Excerpts,Share of Errors,Examples
3,Parse validation failure,6,1,37.5%,E11 (shipments '8.1 Mt' string crashed pydantic)
4,Unit conversion,3,2,18.8%,"E03 (x1,000 instead of x10,000), E19 (short tons to Mt)"
5,Wrong date as net zero,2,2,12.5%,"E10 (2030 intensity target), E20 (2026 first-coil)"
6,Wrong period,2,2,12.5%,"E07 (2023 column used), E08 (2025 fiscal end-year used)"
0,Combined scope as Scope 1,1,1,6.2%,E09 (used combined 8.7 Mt as Scope 1)
1,Computed value not reported,1,1,6.2%,E05 (computed Scope 1 = 2.31 x 18.2 Mt)
2,Non-disclosure as zero,1,1,6.2%,E18 (Scope 2 unquantified reported as 0.0)


**Your answer (Q3):**

* **(a) Overall accuracy and worst field:**
  * **Overall greedy accuracy:** **86.7%** (104 / 120 fields correct).
  * **Worst field:** `scope1_tco2` at **75.0%** (15 / 20 correct; 5 errors).

* **(b) Error taxonomy (16 wrong fields across 10 excerpts):**
  1. **Parse validation failure (6 fields, 1 excerpt):** E11 crashed on `production_mt: "8.1 Mt"`, failing all 6 fields. Note that 8.1 was shipments; liquid steel production was 9.8 Mt.
  2. **Unit conversion (3 fields, 2 excerpts):** E03 applied $\times 1,000$ instead of $\times 10,000$ to Scope 1 (4.108 Mt vs 41.080 Mt) and Scope 2 (0.512 Mt vs 5.120 Mt); E19 extracted 3.3 million short tons directly without converting to metric tonnes (2.99 Mt).
  3. **Wrong date as net zero (2 fields, 2 excerpts):** E10 extracted a 2030 intensity reduction target; E20 extracted 2026 (first-coil commissioning date). Neither is a net-zero target.
  4. **Wrong period (2 fields, 2 excerpts):** E07 took the 2023 column (14.2 Mt) instead of 2024 (13.8 Mt); E08 took fiscal end-year 2025 instead of start-year 2024.
  5. **Computed value not reported (1 field, 1 excerpt):** E05 COMPUTED Scope 1 ($2.31 \text{ t/t} \times 18.2 \text{ Mt} = 42,042,000 \text{ t}$) despite the prompt forbidding computation.
  6. **Combined scope as Scope 1 (1 field, 1 excerpt):** E09 extracted combined Scope 1+2 (8.7 Mt) as Scope 1, violating the null instruction.
  7. **Non-disclosure as zero (1 field, 1 excerpt):** E18 stated Scope 2 would be disclosed in 2025; model recorded `0.0` instead of `null`.
  * *Most frequent:* By **fields**, parse failure dominates (6 fields, 37.5%). By **excerpts**, unit conversion, wrong date, and wrong period are tied (2 excerpts each).

* **(c) Human likelihood:**
  * A human annotator would easily make the **unit conversion errors** (E03 10,000 tonnes, E19 short tons) and **fiscal start-year confusion** (E08). Humans rarely cause schema syntax crashes (E11) or mistake plant commissioning dates for net-zero targets (E20).


## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [9]:
# Compute Bayesian credible interval and bootstrap interval on accuracy
k = int(greedy_scores["correct"].sum())
n = int(len(greedy_scores))

# 1. Bayesian Beta-Binomial: Beta(1 + k, 1 + n - k)
beta_ci = stats.beta(1 + k, 1 + n - k).ppf([0.05, 0.95])
p_gt_95_beta = 1.0 - stats.beta(1 + k, 1 + n - k).cdf(0.95)

# 2. Bootstrap over excerpts (resample 20 excerpts with replacement, 2,000 times)
rng = np.random.default_rng(SEED)
excerpt_summary = greedy_scores.groupby("id")["correct"].agg(["sum", "count"]).reset_index()
N_excerpts = len(excerpt_summary)

boot_accs = []
for _ in range(2000):
    sample_indices = rng.integers(0, N_excerpts, size=N_excerpts)
    sampled = excerpt_summary.iloc[sample_indices]
    boot_accs.append(sampled["sum"].sum() / sampled["count"].sum())

boot_ci = np.percentile(boot_accs, [5, 95])

print(f"k = {k}, n = {n}")
print(f"Beta(1, 1) 90% Credible Interval: [{beta_ci[0]:.3f}, {beta_ci[1]:.3f}] ({beta_ci[0]:.1%}, {beta_ci[1]:.1%}) | Width: {beta_ci[1] - beta_ci[0]:.1%}")
print(f"Bootstrap 90% Confidence Interval: [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}] ({boot_ci[0]:.1%}, {boot_ci[1]:.1%}) | Width: {boot_ci[1] - boot_ci[0]:.1%}")
print(f"P(accuracy > 95% | data) [Beta]:   {p_gt_95_beta:.4f} ({p_gt_95_beta:.2%})")


k = 104, n = 120
Beta(1, 1) 90% Credible Interval: [0.806, 0.908] (80.6%, 90.8%) | Width: 10.2%
Bootstrap 90% Confidence Interval: [0.775, 0.933] (77.5%, 93.3%) | Width: 15.8%
P(accuracy > 95% | data) [Beta]:   0.0001 (0.01%)


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

* **(a) Reported intervals and why bootstrap is wider:**
  * **Beta-Binomial 90% Credible Interval:** **[80.6%, 90.8%]** (width: 10.2 pp).
  * **Bootstrap 90% Confidence Interval (2,000 resamples):** **[77.5%, 93.3%]** (width: 15.8 pp).
  * *Why bootstrap is wider:* Beta-Binomial assumes all $n=120$ fields are independent Bernoulli trials ($i.i.d.$). In reality, the 6 fields within an excerpt are **strongly clustered** (e.g., when E11 fails parsing, 6 fields fail together). The effective sample size is close to 20 excerpts, not 120 fields. Resampling whole excerpts preserves this intra-document correlation. Note that bootstrap with only 20 clusters is somewhat rough.

* **(b) Sample size to halve width:**
  * Because standard error decays as $1/\sqrt{N}$, halving the interval width requires **$4\times$ the sample size** ($\approx 80$ excerpts).

* **(c) Consistency with vendor claim of 95%:**
  * **Inconsistent for greedy extraction.** 95% lies above both 90% intervals; the posterior probability of greedy accuracy exceeding 95% is only **0.01%** ($P = 0.0001$).
  * However, our **5-sample voting pipeline** achieves **95.0%** (Beta CI: [90.4%, 97.3%]), making the claim plausible under self-consistency.


## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [10]:
# Downstream analysis: Portfolio Scope 1 total and median emission intensity
downstream = pd.DataFrame(index=gold.index)
greedy_idx = greedy_pred.set_index("id")

downstream["scope1_llm"] = greedy_idx["scope1_tco2"]
downstream["scope1_gold"] = gold["scope1_tco2"]
downstream["prod_llm"] = greedy_idx["production_mt"]
downstream["prod_gold"] = gold["production_mt"]

# Treat missing values as 0 for portfolio total
downstream["scope1_llm_filled"] = downstream["scope1_llm"].fillna(0)
downstream["scope1_gold_filled"] = downstream["scope1_gold"].fillna(0)
downstream["error_t"] = downstream["scope1_llm_filled"] - downstream["scope1_gold_filled"]

# Compute portfolio totals (in MtCO2e)
total_llm_mt = downstream["scope1_llm_filled"].sum() / 1e6
total_gold_mt = downstream["scope1_gold_filled"].sum() / 1e6
net_error_mt = downstream["error_t"].sum() / 1e6
abs_error_mt = downstream["error_t"].abs().sum() / 1e6

# Median emission intensity (tCO2 / tonne of steel = tCO2 / (Mt * 1e6))
# Computed where both scope1 and production are reported
int_llm = downstream["scope1_llm"] / (downstream["prod_llm"] * 1e6)
int_gold = downstream["scope1_gold"] / (downstream["prod_gold"] * 1e6)
valid_llm = int_llm.dropna()
valid_gold = int_gold.dropna()
common_idx = valid_llm.index.intersection(valid_gold.index)

print(f"Scope 1 Total (LLM):  {total_llm_mt:,.3f} MtCO2e")
print(f"Scope 1 Total (Gold): {total_gold_mt:,.3f} MtCO2e")
print(f"Net Error:            {net_error_mt:+,.3f} MtCO2e ({net_error_mt / total_gold_mt:+.1%})")
print(f"Absolute Error:       {abs_error_mt:,.3f} MtCO2e ({abs_error_mt / total_gold_mt:.1%})")
print(f"Median Intensity (LLM, n={len(valid_llm)}):   {valid_llm.median():.3f} tCO2 / t steel")
print(f"Median Intensity (Gold, n={len(valid_gold)}):  {valid_gold.median():.3f} tCO2 / t steel")
print(f"Median Intensity (Common, n={len(common_idx)}): LLM={int_llm.loc[common_idx].median():.3f} vs Gold={int_gold.loc[common_idx].median():.3f} tCO2 / t steel")

print("\nTop contributing excerpts to Scope 1 error:")
display(downstream[["scope1_llm", "scope1_gold", "error_t"]]
        .sort_values(by="error_t", key=abs, ascending=False).head(5))


Scope 1 Total (LLM):  252.112 MtCO2e
Scope 1 Total (Gold): 242.062 MtCO2e
Net Error:            +10.050 MtCO2e (+4.2%)
Absolute Error:       92.234 MtCO2e (38.1%)
Median Intensity (LLM, n=18):   1.895 tCO2 / t steel
Median Intensity (Gold, n=17):  1.865 tCO2 / t steel
Median Intensity (Common, n=16): LLM=1.845 vs Gold=1.878 tCO2 / t steel

Top contributing excerpts to Scope 1 error:


,scope1_llm,scope1_gold,error_t
id,,,
E05,"42,042,000.000",NaN,"42,042,000.000"
E03,"4,108,000.000","41,080,000.000","-36,972,000.000"
E09,"8,700,000.000",NaN,"8,700,000.000"
E11,NaN,"4,120,000.000","-4,120,000.000"
E07,"14,200,000.000","13,800,000.000","400,000.000"


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

* **(a) Portfolio error metrics and top two contributors:**
  * **LLM Total:** **252.112 MtCO2e** vs **Gold Total:** **242.062 MtCO2e**.
  * **Net error:** **+10.050 MtCO2e** (+4.2%).
  * **Absolute error:** **92.234 MtCO2e** (**38.1%** gross distortion).
  * **Top two contributors:**
    1. **E05:** **+42.042 MtCO2e** (model computed Scope 1 from intensity $\times$ production; gold is null).
    2. **E03:** **-36.972 MtCO2e** (applied $\times 1,000$ instead of $\times 10,000$; extracted 4.108 Mt vs 41.080 Mt).

* **(b) Why 90% field accuracy hides massive portfolio error, while the median barely moves:**
  * *The mean hides the tail:* Portfolio totals are additive sums. Huge opposing tail errors (+42 Mt on E05 and -37 Mt on E03) cancel each other out, masking a massive **92.2 Mt absolute error** behind a benign +10.1 Mt net total.
  * *Median stability:* The median intensity (1.895 vs 1.865 across all available; 1.845 vs 1.878 on the 16 common companies) is an order statistic. Both major errors fall on the same side of the median as their true values (E03 intensity 0.19 is below median; E05 2.31 is above median), leaving the central rank unchanged.

* **(c) What to report to the lender:**
  1. The **92.2 Mt is a sum of absolute errors, NOT a confidence interval**; it represents total gross misallocated capital risk.
  2. Data coverage and missingness: list companies where emissions could not be verified.
  3. Mandatory manual review of top emitters: since portfolio emissions follow a power law, auditing the top 20% of facilities mitigates >80% of tail error.


---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [11]:
# Self-consistency: majority vote and agreement across 5 samples
sample_preds = pred[pred["kind"] == "sample"].copy()

votes_records = []
for eid in gold.index:
    eid_samples = sample_preds[sample_preds["id"] == eid]
    n_samples = len(eid_samples)
    for f in FIELDS:
        g_val = gold.loc[eid, f]
        if n_samples == 0:
            votes_records.append({"id": eid, "field": f, "vote": None, "agreement": 0.0, "correct": False})
            continue

        # Convert values to JSON-serializable keys (handling None, NaN, and int64)
        def to_key(val):
            if pd.isna(val):
                return None
            if hasattr(val, "item"):
                return val.item()
            return val

        raw_vals = eid_samples[f].values
        str_keys = [json.dumps(to_key(v)) for v in raw_vals]
        counts = pd.Series(str_keys).value_counts()
        winner_key = counts.index[0]
        winner_count = counts.iloc[0]
        winner_val = json.loads(winner_key)
        agreement = winner_count / 5.0

        corr = field_correct(f, winner_val, g_val)
        votes_records.append({
            "id": eid,
            "field": f,
            "vote": winner_val,
            "agreement": agreement,
            "correct": corr
        })

votes = pd.DataFrame(votes_records)

# Agreement bin assignment
def assign_bin(agr):
    if agr >= 1.0:
        return "5/5"
    if agr >= 0.8:
        return "4/5"
    return "<=3/5"

votes["agreement_bin"] = votes["agreement"].apply(assign_bin)

# Accuracy comparison
vote_accuracy = votes["correct"].mean()
print(f"Greedy Accuracy: {overall_accuracy:.1%} | Majority Vote Accuracy: {vote_accuracy:.1%}")

# Accuracy per agreement bin
bin_table = (votes.groupby("agreement_bin")["correct"]
             .agg(total="count", correct="sum", accuracy="mean")
             .loc[["5/5", "4/5", "<=3/5"]])
display(bin_table.style.format({"accuracy": "{:.1%}"}))


Greedy Accuracy: 86.7% | Majority Vote Accuracy: 95.0%


,total,correct,accuracy
agreement_bin,,,
5/5,93,91,97.8%
4/5,15,14,93.3%
<=3/5,12,9,75.0%


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [12]:
# Detailed calculations for Q6: bin breakdown, Beta intervals, and probabilities
k_5 = int(bin_table.loc["5/5", "correct"])
n_5 = int(bin_table.loc["5/5", "total"])
beta_5_ci = stats.beta(1 + k_5, 1 + n_5 - k_5).ppf([0.05, 0.95])
p_gt_95_vote5 = 1.0 - stats.beta(1 + k_5, 1 + n_5 - k_5).cdf(0.95)

k_vote = int(votes["correct"].sum())
n_vote = len(votes)
vote_beta_ci = stats.beta(1 + k_vote, 1 + n_vote - k_vote).ppf([0.05, 0.95])

print(f"5/5 agreement bin: {k_5}/{n_5} correct ({k_5/n_5:.1%})")
print(f"Beta(1, 1) 90% Credible Interval for 5/5: [{beta_5_ci[0]:.3f}, {beta_5_ci[1]:.3f}] ({beta_5_ci[0]:.1%}, {beta_5_ci[1]:.1%})")
print(f"P(accuracy > 95% | 5/5 bin) [Beta]:        {p_gt_95_vote5:.4f} ({p_gt_95_vote5:.1%})")
print(f"Overall Vote Beta(1, 1) 90% Credible Interval: [{vote_beta_ci[0]:.3f}, {vote_beta_ci[1]:.3f}] ({vote_beta_ci[0]:.1%}, {vote_beta_ci[1]:.1%})")
print(f"Greedy accuracy without E11 parse failure: {104}/114 = {104/114:.1%}")


5/5 agreement bin: 91/93 correct (97.8%)
Beta(1, 1) 90% Credible Interval for 5/5: [0.935, 0.991] (93.5%, 99.1%)
P(accuracy > 95% | 5/5 bin) [Beta]:        0.8546 (85.5%)
Overall Vote Beta(1, 1) 90% Credible Interval: [0.904, 0.973] (90.4%, 97.3%)
Greedy accuracy without E11 parse failure: 104/114 = 91.2%


**Your answer (Q6):**

* **(a) Accuracy comparison and agreement bins:**
  * **Greedy accuracy:** **86.7%** (104 / 120). (Excluding E11's syntax crash: 104 / 114 = **91.2%**).
  * **Majority vote accuracy:** **95.0%** (114 / 120; 90% Beta CI: [90.4%, 97.3%]).
  * **By agreement bin:**
    * **5/5 agreement:** **97.8%** (91 / 93 correct; error rate 2.2%).
    * **4/5 agreement:** **93.3%** (14 / 15 correct; error rate 6.7%).
    * **$\le$3/5 agreement:** **75.0%** (9 / 12 correct; error rate 25.0%).

* **(b) Low agreement as an alarm:**
  * **Yes, highly effective.** The error rate surges more than tenfold from 2.2% (at 5/5) to 25.0% (at $\le$3/5). Disagreement is a clear symptom of model hesitation and source ambiguity.

* **(c) Can we claim 5/5 accuracy is above 95%?**
  * **No.** With 91/93 correct, Beta(92, 3) gives a 90% credible interval of **[93.5%, 99.1%]**. Because the lower bound dips to 93.5%, we cannot guarantee $>95\%$ at 90% confidence, even though the point estimate is 97.8% and $P(\text{acc} > 0.95) \approx 85\%$.


## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [13]:
# Identify confidently wrong fields: 5/5 agreement but incorrect
confidently_wrong = votes[(votes["agreement"] == 1.0) & (~votes["correct"])].copy()
confidently_wrong["gold"] = confidently_wrong.apply(lambda r: gold.loc[r["id"], r["field"]], axis=1)

print(f"Confidently wrong fields (agreement = 5/5, correct = False): {len(confidently_wrong)}")
display(confidently_wrong[["id", "field", "vote", "gold"]])

# Print relevant excerpt text for qualitative inspection
for eid in confidently_wrong["id"].unique():
    print(f"\n--- Excerpt {eid} ---")
    print(excerpts.loc[excerpts["id"] == eid, "text"].values[0].strip())


Confidently wrong fields (agreement = 5/5, correct = False): 2


,id,field,vote,gold
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000"
112,E19,production_mt,3.300,2.990



--- Excerpt E03 ---
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

--- Excerpt E19 ---
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.


**Your answer (Q7):**

* **(a) What the confident errors have in common:**
  * Two fields agree 5/5 but are wrong: **E03 scope1_tco2** (vote `4,108,000` vs gold `41,080,000`) and **E19 production_mt** (vote `3.3` vs gold `2.99`).
  * Both are **unit conversion blind spots**: the model extracts the literal surface numeral ($4,108$ and $3.3$) perfectly, but repeatedly ignores domain unit multipliers ("10,000 tonnes" and "short tons").

* **(b) Bias (epistemic) vs noise (aleatoric) & why resampling fails:**
  * Agreement measures **consistency, not correctness**. Temperature sampling explores token noise around the model's learned distribution (aleatoric uncertainty). When the model has a systematic blind spot (epistemic bias), every sample reproduces the exact same mistake with 100% agreement.
  * *Fixes:* Resampling cannot fix systematic bias. Solutions require:
    1. A deterministic **regex unit trigger + human review**.
    2. A **second, architecturally distinct model** (helps only if the blind spot is not shared).
    3. Targeted prompt exemplars (must be tested out-of-sample).

* **(c) Analogue in Session 4:**
  * Corresponds to a **common-mode bias** / systematic instrument calibration error (e.g. an incorrect default emission factor or miscalibrated sensor applied across all plants, where averaging multiple readings reproduces the bias).


## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [14]:
# Analyze model's self-reported confidence calibration on sample responses
sample_eval = []
for (eid, s_idx), grp in sample_preds.groupby(["id", "sample"]):
    all_correct = True
    for f in FIELDS:
        pval = grp[f].values[0]
        gval = gold.loc[eid, f]
        if not field_correct(f, pval, gval):
            all_correct = False
            break
    conf = grp["confidence"].values[0]
    sample_eval.append({"id": eid, "sample": s_idx, "confidence": conf, "all_correct": all_correct})

calib_df = pd.DataFrame(sample_eval)

mean_conf = calib_df["confidence"].mean()
share_all_corr = calib_df["all_correct"].mean()
print(f"Mean Stated Confidence:            {mean_conf:.1%}")
print(f"Share with All 6 Fields Correct:   {share_all_corr:.1%} ({calib_df['all_correct'].sum()}/{len(calib_df)})")

# Binning by confidence intervals
calib_df["conf_bin"] = pd.cut(calib_df["confidence"], bins=[0.0, 0.90, 0.93, 0.96, 1.0])
calib_table = (calib_df.groupby("conf_bin", observed=False)["all_correct"]
               .agg(total="count", fully_correct="sum", accuracy="mean"))
display(calib_table.style.format({"accuracy": "{:.1%}"}))

# Note overconfidence on problematic excerpts
for eid in ["E03", "E09", "E19"]:
    e_confs = calib_df[calib_df["id"] == eid]["confidence"].tolist()
    print(f"Excerpt {eid} stated confidences across 5 samples: {e_confs} (none fully correct)")


Mean Stated Confidence:            92.8%
Share with All 6 Fields Correct:   55.0% (55/100)


,total,fully_correct,accuracy
conf_bin,,,
"(0.0, 0.9]",21,6,28.6%
"(0.9, 0.93]",35,16,45.7%
"(0.93, 0.96]",38,28,73.7%
"(0.96, 1.0]",6,5,83.3%


Excerpt E03 stated confidences across 5 samples: [0.89, 0.89, 0.91, 0.93, 0.94] (none fully correct)
Excerpt E09 stated confidences across 5 samples: [0.92, 0.91, 0.93, 0.95, 0.9] (none fully correct)
Excerpt E19 stated confidences across 5 samples: [0.91, 0.94, 0.85, 0.93, 0.95] (none fully correct)


**Your answer (Q8):**

* **Is the model over-confident?**
  * **Severely over-confident.** Mean stated confidence is **92.8%**, but only **55.0%** of sample responses (55 / 100) have all six fields correct (a 37.8 pp calibration gap).
  * Stated confidence is 0.85–0.95 even on excerpts like E03, E09, and E19 where **zero** sample responses were fully correct.

* **Is stated confidence useful for ranking?**
  * **Yes, monotonically informative for sorting:**
    * $\le 0.90$: 28.6% fully correct (6 / 21)
    * $0.90 - 0.93$: 45.7% fully correct (16 / 35)
    * $0.93 - 0.96$: 73.7% fully correct (28 / 38)
    * $> 0.96$: 83.3% fully correct (5 / 6; note small $n=6$)
  * Stated confidence works as an ordinal priority score for routing, but never as an absolute probability of correctness.


## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [15]:
# Evaluate review policies: P0, P1, P2, and P3 (extension)
# Unit costs: Review = EUR 4 / field; Escaped error = EUR 150 / field
COST_REVIEW = 4
COST_ERROR = 150

# Unit trigger for extension P3: detect non-standard unit mentions in excerpts
unit_pattern = re.compile(r"short tons?|10,000 tonnes", re.IGNORECASE)
unit_flagged_eids = set(excerpts[excerpts["text"].str.contains(unit_pattern)]["id"])

policy_conditions = [
    ("P0: Trust vote everywhere", lambda r: False),
    ("P1: Review agreement < 5/5", lambda r: r["agreement"] < 1.0),
    ("P2: Review agreement <= 3/5", lambda r: r["agreement"] <= 0.6),
    ("P3: P1 + Unit trigger (regex)", lambda r: (r["agreement"] < 1.0) or (r["id"] in unit_flagged_eids))
]

policy_results = []
for name, condition in policy_conditions:
    needs_review = votes.apply(condition, axis=1)
    n_reviewed = int(needs_review.sum())
    surviving_errors = int((~needs_review & ~votes["correct"]).sum())
    cost_rev = n_reviewed * COST_REVIEW
    cost_err = surviving_errors * COST_ERROR
    total_cost = cost_rev + cost_err
    policy_results.append({
        "Policy": name,
        "Reviewed Fields": n_reviewed,
        "Residual Errors": surviving_errors,
        "Review Cost (EUR)": cost_rev,
        "Error Cost (EUR)": cost_err,
        "Total Cost (EUR)": total_cost
    })

policy_df = pd.DataFrame(policy_results).set_index("Policy")
display(policy_df)

# Print break-even economics and error rates by bin
break_even_rate = COST_REVIEW / COST_ERROR
print(f"\nBreak-even error rate: EUR {COST_REVIEW} / EUR {COST_ERROR} = {break_even_rate:.1%}")
for b in ["5/5", "4/5", "<=3/5"]:
    b_err = 1.0 - bin_table.loc[b, "accuracy"]
    print(f"Error rate in bin {b:>5}: {b_err:.1%} (vs break-even {break_even_rate:.1%})")


,Reviewed Fields,Residual Errors,Review Cost (EUR),Error Cost (EUR),Total Cost (EUR)
Policy,,,,,
P0: Trust vote everywhere,0,6,0,900,900
P1: Review agreement < 5/5,27,2,108,300,408
P2: Review agreement <= 3/5,12,3,48,450,498
P3: P1 + Unit trigger (regex),37,0,148,0,148



Break-even error rate: EUR 4 / EUR 150 = 2.7%
Error rate in bin   5/5: 2.2% (vs break-even 2.7%)
Error rate in bin   4/5: 6.7% (vs break-even 2.7%)
Error rate in bin <=3/5: 25.0% (vs break-even 2.7%)


**Your answer (Q9):**

* **Cost breakdown and policy choice:**
  * **Break-even error rate:** $\text{EUR } 4 / \text{EUR } 150 = \mathbf{2.7\%}$.
  * **P0 (Trust vote):** 0 reviewed, 6 errors left $\to$ **EUR 900**.
  * **P1 (Review agreement < 5/5):** 27 reviewed, 2 errors left $\to$ **EUR 408** (EUR 108 review + EUR 300 error).
  * **P2 (Review agreement $\le$ 3/5):** 12 reviewed, 3 errors left $\to$ **EUR 498** (EUR 48 review + EUR 450 error).
  * **P1 beats P2** because bin 4/5 has an error rate of 6.7%, well above the 2.7% break-even threshold. Reviewing bin 4/5 costs EUR 4 to eliminate an expected EUR 10 error penalty, generating net savings.
  * *Voting cost:* 5 sample calls multiply the LLM API bill by $5\times$ compared to P0.

* **Surviving errors and Extension P3:**
  * P1 leaves the 2 confidently wrong errors: E03 Scope 1 (37 Mt error) and E19 production.
  * **Extension P3 (P1 + unit trigger `short tons?|10,000 tonnes`):** Reviews 37 fields, leaves **0 errors**, total cost **EUR 148**.
  * *Limitations:*
    1. **In-sample tuning:** P3 is tuned on the same 20 excerpts; in production, regex will miss unseen units ("kt", "lbs", "thousand tonnes").
    2. **Symmetric cost limitation:** The EUR 150 average penalty treats all errors equally, but E03's 37 Mt error is catastrophic.
    3. **Smart next step:** Weight review priority by $\text{agreement} \times \text{emissions}$ to audit mega-emitters first.


---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [16]:
# Part D1: TF-IDF Retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = corpus["chunks"]
queries = corpus["queries"]

vectorizer = TfidfVectorizer(stop_words="english")
X_chunks = vectorizer.fit_transform(chunks)
X_queries = vectorizer.transform([q["query"] for q in queries])
tfidf_sims = cosine_similarity(X_queries, X_chunks)

retrieval_records = []
for i, q in enumerate(queries):
    top3 = np.argsort(-tfidf_sims[i])[:3].tolist()
    rel = q["relevant"][0]
    h1 = (top3[0] == rel)
    h3 = (rel in top3)
    retrieval_records.append({
        "qid": q["qid"],
        "query": q["query"],
        "relevant_chunk": rel,
        "top3_chunks": top3,
        "hit@1": h1,
        "hit@3": h3
    })

retrieval = pd.DataFrame(retrieval_records)
print(f"TF-IDF hit@1: {retrieval['hit@1'].mean():.1%} ({retrieval['hit@1'].sum()}/{len(retrieval)})")
print(f"TF-IDF hit@3: {retrieval['hit@3'].mean():.1%} ({retrieval['hit@3'].sum()}/{len(retrieval)})")
display(retrieval)


TF-IDF hit@1: 66.7% (4/6)
TF-IDF hit@3: 66.7% (4/6)


,qid,query,relevant_chunk,top3_chunks,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,3,"[1, 5, 12]",False,False
1,Q2,net zero target year,5,"[5, 0, 13]",True,True
2,Q3,crude steel production 2024,2,"[2, 3, 0]",True,True
3,Q4,investment in hydrogen DRI capex,6,"[6, 0, 2]",True,True
4,Q5,impact of the carbon price on EBITDA,8,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,9,"[3, 6, 1]",False,False


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [17]:
# Inspect failed TF-IDF queries: analyze tokenization, scores, and chunk ranks
failed_queries = retrieval[~retrieval["hit@1"]]
analyzer = vectorizer.build_analyzer()

for _, r in failed_queries.iterrows():
    qid = r["qid"]
    rel_idx = r["relevant_chunk"]
    ret_idx = r["top3_chunks"][0]
    q_tokens = analyzer(r["query"])
    rel_tokens = analyzer(chunks[rel_idx])
    ret_tokens = analyzer(chunks[ret_idx])
    
    # Find rank of relevant chunk
    q_idx = int(qid[1:]) - 1
    all_ranks = np.argsort(-tfidf_sims[q_idx]).tolist()
    rel_rank = all_ranks.index(rel_idx) + 1
    
    print(f"================ Query {qid} ================")
    print(f"Query: \"{r['query']}\"")
    print(f"Tokens extracted by TF-IDF analyzer: {q_tokens}")
    print(f"[Rank 1 Retrieved Chunk #{ret_idx}] (sim={tfidf_sims[q_idx, ret_idx]:.3f}):")
    print(f"  Text: {chunks[ret_idx].strip()[:140]}...")
    print(f"  Shared tokens with query: {set(q_tokens).intersection(ret_tokens)}")
    print(f"[Ground-Truth Chunk #{rel_idx}] (Rank #{rel_rank}, sim={tfidf_sims[q_idx, rel_idx]:.3f}):")
    print(f"  Text: {chunks[rel_idx].strip()[:140]}...")
    print(f"  Shared tokens with query: {set(q_tokens).intersection(rel_tokens)}")
    print("-" * 60 + "\n")


================ Query Q1 ================
Query: "What were Scope 1 emissions in 2024?"
Tokens extracted by TF-IDF analyzer: ['scope', 'emissions', '2024']
[Rank 1 Retrieved Chunk #1] (sim=0.338):
  Text: About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 cover...
  Shared tokens with query: {'emissions', 'scope'}
[Ground-Truth Chunk #3] (Rank #4, sim=0.184):
  Text: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an...
  Shared tokens with query: {'emissions', '2024'}
------------------------------------------------------------

================ Query Q6 ================
Query: "flooding exposure of our plants"
Tokens extracted by TF-IDF analyzer: ['flooding', 'exposure', 'plants']
[Rank 1 Retrieved Chunk #3] (sim=0.208):
  Text: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke o

**Your answer (Q10):**

* **(a) Retrieval metrics:**
  * **hit@1:** **66.7%** (4 / 6). **hit@3:** **66.7%** (4 / 6).

* **(b) Why TF-IDF failed on Q1 and Q6:**
  * **Q1 ("What were Scope 1 emissions in 2024?"):** Default vectorizer drops single-character tokens (`token_pattern=r"(?u)\b\w\w+\b"`), stripping "1" from "Scope 1" $\to$ `['scope', 'emissions', '2024']`. Scope 1, 2, and 3 become identical. Chunk 1 (methodology) wins on term frequency; relevant chunk 3 ranks 4th because it uses "direct greenhouse gas emissions" without the token "Scope 1".
  * **Q6 ("flooding exposure of our plants"):** Pure **vocabulary mismatch**. Chunk 9 uses "riverine inundation" and "sites", sharing zero tokens with the query. Chunk 3 wins Rank 1 by accident via the single shared word "plants".

* **(c) What dense embeddings fix:**
  * Embeddings resolve **Q6**, projecting "flooding" and "riverine inundation" into the same semantic space.
  * Embeddings do **NOT** fix Q1 (chunk 5 targets still beats chunk 3), because understanding that Scope 1 means direct emissions requires domain grounding.

* **(d) Why retrieval misses are more dangerous than extraction errors:**
  * Retrieval is a **hard bottleneck**. If the right chunk is missed, the LLM receives zero relevant context. The failure is **silent AND systematic**: all 5 resampled LLM calls see the same wrong chunk, so majority voting cannot detect or fix it.


> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [18]:
# Optional D2: Dense embeddings retrieval using SentenceTransformer
try:
    from sentence_transformers import SentenceTransformer
    dense_model = SentenceTransformer("all-MiniLM-L6-v2")

    chunk_embeddings = dense_model.encode(chunks)
    query_embeddings = dense_model.encode([q["query"] for q in queries])
    dense_sims = cosine_similarity(query_embeddings, chunk_embeddings)

    dense_records = []
    for i, q in enumerate(queries):
        top3 = np.argsort(-dense_sims[i])[:3].tolist()
        rel = q["relevant"][0]
        h1 = (top3[0] == rel)
        h3 = (rel in top3)
        dense_records.append({
            "qid": q["qid"],
            "query": q["query"],
            "relevant_chunk": rel,
            "top3_chunks": top3,
            "hit@1": h1,
            "hit@3": h3
        })

    dense_retrieval = pd.DataFrame(dense_records)
    print(f"Dense Embeddings hit@1: {dense_retrieval['hit@1'].mean():.1%} ({dense_retrieval['hit@1'].sum()}/{len(dense_retrieval)})")
    print(f"Dense Embeddings hit@3: {dense_retrieval['hit@3'].mean():.1%} ({dense_retrieval['hit@3'].sum()}/{len(dense_retrieval)})")
    display(dense_retrieval)

    # Check which query still fails
    still_failed = dense_retrieval[~dense_retrieval["hit@1"]]
    print("\nQueries still failing under dense embeddings:")
    for _, r in still_failed.iterrows():
        print(f"- {r['qid']}: \"{r['query']}\" -> retrieved chunk {r['top3_chunks'][0]} (targets) instead of chunk {r['relevant_chunk']} (climate performance)")

except Exception as e:
    print(f"Dense embeddings note: Model download / execution unavailable ({e}). Keeping documented reference output.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense Embeddings hit@1: 83.3% (5/6)
Dense Embeddings hit@3: 83.3% (5/6)


,qid,query,relevant_chunk,top3_chunks,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,3,"[5, 12, 1]",False,False
1,Q2,net zero target year,5,"[5, 0, 6]",True,True
2,Q3,crude steel production 2024,2,"[2, 0, 3]",True,True
3,Q4,investment in hydrogen DRI capex,6,"[6, 8, 7]",True,True
4,Q5,impact of the carbon price on EBITDA,8,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,9,"[9, 10, 3]",True,True



Queries still failing under dense embeddings:
- Q1: "What were Scope 1 emissions in 2024?" -> retrieved chunk 5 (targets) instead of chunk 3 (climate performance)


---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

We evaluated an LLM extraction pipeline on steel disclosures. Single-run accuracy is 87% (90% range: 81–91%), rising to 95% under 5-sample voting. However, the model repeatedly misses non-standard units (e.g. short tons), making voting blind to large errors.

We recommend adopting the pipeline with human review for disagreements and flagged units. This costs about EUR 150 per 120 fields and is *likely* to prevent material misstatements. A key limitation is our sample of only 20 reports, leaving performance on varied disclosure styles unverified.
